# Changelog checkpoints

The file at https://taurus.agglomeration.ariadne.ai/api/changes.tsv contains the following (from https://taurus.agglomeration.ariadne.ai/api/)


    GET changes.tsv file that contains chronologically sorted changes in ascending order as a tab-separated text file. List of changes with corresponding row format:

    add edge:
        +\t[supervoxel id 1]\t[supervoxel id 2]
    delete edge:
        -\t[supervoxel id 1]\t[supervoxel id 2]
    add anchor:
        anchor_add\t[anchor id]
    delete added anchor:
        anchor_del\t[anchor id]
    set anchor note:
        note\t[anchor id]\t[comment] 


In [1]:
# !wget https://taurus.agglomeration.ariadne.ai/api/changes.tsv

In [2]:
import networkx as nx


def parse_changes_to_nx(file_path: str, first_n_changes: int | None = None):
    G = nx.Graph()

    with open(file_path, "r") as file:
        lines = []
        for line in file:
            lines.append(line)
            parts = line.strip().split("\t")
            if parts[0] == "+":
                G.add_edge(parts[1], parts[2])
            elif parts[0] == "-":
                G.remove_edge(parts[1], parts[2])
            elif parts[0] == "anchor_add":
                G.add_node(parts[1], type="anchor")
            elif parts[0] == "anchor_del":
                if G.has_node(parts[1]):
                    G.remove_node(parts[1])
            elif parts[0] == "note":
                if G.has_node(parts[1]):
                    G.nodes[parts[1]]["note"] = parts[2]
            if first_n_changes is not None and len(lines) >= first_n_changes:
                break
    with open(f"component-graphs/{file_path}.{len(lines)}", "w") as fh:
        fh.writelines(lines)
    return G, len(lines)

In [3]:
_, current_length = parse_changes_to_nx("changes.tsv")
print(current_length)

48700


In [4]:
for i in range(0, current_length, 5_000):
    if i == 0:
        continue
    g, length = parse_changes_to_nx("changes.tsv", i)
    nx.write_gexf(g, f"component-graphs/change_nx.{length}.gexf")